# Crop Disease Detection — EfficientNet Fine-Tuning (Colab, L4)

End-to-end pipeline that produces a **drop-in ONNX model + metadata bundle** for the
FastAPI `InferenceProvider` seam in your backend.

**Flow:** download → augment → 2-phase fine-tune → evaluate → export ONNX → package `model/` bundle → sanity-test against the backend contract.

- **Framework:** PyTorch + `timm`
- **Model:** EfficientNet-B3 (300px), one-line switch to B4 (380px)
- **Server target:** ONNX via `onnxruntime==1.19.2`
- **Master artifact:** best torch checkpoint (kept on Drive; TFLite path stays open via ONNX)

---
### ⚠️ Security — your Kaggle key was shared in plaintext
Treat it as compromised. After this runs, go to **Kaggle → Settings → API → Expire API Token** and generate a fresh one.
This notebook reads the key from **Colab Secrets** (or a manual upload) so the leaked value is never written into your repo.


## 1. Runtime check
Confirm you're on the L4 GPU before training.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "No GPU — set Runtime > Change runtime type > L4 GPU")
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2. Install dependencies
`torch` ships with Colab. We pin `onnxruntime==1.19.2` to match the backend.

In [ ]:
%pip install -q "timm==1.0.11" "onnx==1.16.2" "onnxruntime==1.19.2" "onnxsim==0.4.36" "kaggle==1.6.17" scikit-learn matplotlib
print("deps installed")

## 3. Config — one place for every knob
**Switch to B4** by changing `MODEL_KEY = "b4"`. Everything else (input size, batch) follows automatically.

In [ ]:
from dataclasses import dataclass, field

MODELS = {
    # timm noisy-student weights transfer very well for leaf textures
    "b3": {"timm_name": "tf_efficientnet_b3.ns_jft_in1k", "img_size": 300, "batch": 32},
    "b4": {"timm_name": "tf_efficientnet_b4.ns_jft_in1k", "img_size": 380, "batch": 24},
}

@dataclass
class Config:
    MODEL_KEY: str = "b3"                 # <-- change to "b4" to escalate
    seed: int = 42
    # two-phase schedule
    epochs_head: int = 3                  # Phase A: train classifier head only
    epochs_finetune: int = 22             # Phase B: unfreeze + fine-tune
    lr_head: float = 1e-3
    lr_finetune: float = 3e-4
    weight_decay: float = 1e-4
    label_smoothing: float = 0.1          # your proven setting
    num_workers: int = 2
    # imbalance handling
    imbalance_ratio_trigger: float = 1.5  # if max/min class count exceeds this, apply weighting
    # paths
    drive_root: str = "/content/drive/MyDrive/crop_disease"
    data_root: str = "/content/data"      # dataset unzips here (local SSD = fast)

    @property
    def m(self): return MODELS[self.MODEL_KEY]
    @property
    def img_size(self): return self.m["img_size"]
    @property
    def batch(self): return self.m["batch"]
    @property
    def ckpt_dir(self): return f"{self.drive_root}/checkpoints_{self.MODEL_KEY}"
    @property
    def out_dir(self): return f"{self.drive_root}/model_{self.MODEL_KEY}"

cfg = Config()
import torch, random, numpy as np
random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed); torch.cuda.manual_seed_all(cfg.seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Model={cfg.m['timm_name']} | img={cfg.img_size} | batch={cfg.batch} | device={device}")

## 4. Kaggle auth (secure)
**Preferred:** add your key to Colab Secrets — 🔑 icon in the left sidebar → add `KAGGLE_USERNAME` and `KAGGLE_KEY`.
**Fallback:** run the cell and upload your `kaggle.json` when prompted.

Either way the key is *not* saved into any file you'd commit.

In [ ]:
import os, json, stat

def setup_kaggle():
    # 1) Colab Secrets (recommended)
    try:
        from google.colab import userdata
        u, k = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")
        if u and k:
            os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = u, k
            print("Using Kaggle creds from Colab Secrets."); return
    except Exception:
        pass
    # 2) Manual upload fallback
    from google.colab import files
    print("Upload your kaggle.json:")
    up = files.upload()
    fn = next(iter(up))
    os.makedirs("/root/.kaggle", exist_ok=True)
    with open("/root/.kaggle/kaggle.json","wb") as f: f.write(up[fn])
    os.chmod("/root/.kaggle/kaggle.json", stat.S_IRUSR | stat.S_IWUSR)
    print("kaggle.json installed.")

setup_kaggle()

## 5. Download dataset — *New Plant Diseases (Augmented)*
38 classes, ~87k images, pre-split `train/valid` (PlantVillage taxonomy). Unzips to local SSD for fast epochs.

In [ ]:
import os, glob, subprocess

os.makedirs(cfg.data_root, exist_ok=True)
zip_path = f"{cfg.data_root}/npd.zip"
if not glob.glob(f"{cfg.data_root}/**/train", recursive=True):
    subprocess.run(["kaggle","datasets","download","-d","vipoooool/new-plant-diseases-dataset",
                    "-p", cfg.data_root], check=True)
    z = glob.glob(f"{cfg.data_root}/*.zip")[0]
    print("unzipping", z)
    subprocess.run(["unzip","-q","-o", z, "-d", cfg.data_root], check=True)

# locate the actual train/valid dirs (dataset nests them a couple levels deep)
train_dir = sorted(glob.glob(f"{cfg.data_root}/**/train", recursive=True), key=len)[0]
valid_dir = sorted(glob.glob(f"{cfg.data_root}/**/valid", recursive=True), key=len)[0]
print("train:", train_dir)
print("valid:", valid_dir)
print("num class folders:", len(os.listdir(train_dir)))

## 6. Mount Drive for checkpoints
So a Colab disconnect never costs you progress.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(cfg.ckpt_dir, exist_ok=True)
os.makedirs(cfg.out_dir, exist_ok=True)
print("checkpoints ->", cfg.ckpt_dir)

## 7. Transforms & loaders
Preprocessing is pulled straight from the model's own `timm` data config, so the **exact same mean/std/size/interpolation** get baked into the metadata bundle the server uses. Train transforms add strong augmentation (RandAugment + flips + erasing) — this is your main in-pipeline lever for real-world field robustness.

In [ ]:
import timm
from timm.data import resolve_model_data_config, create_transform
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# a throwaway model instance just to resolve the canonical data config
_probe = timm.create_model(cfg.m["timm_name"], pretrained=False, num_classes=1)
data_cfg = resolve_model_data_config(_probe)
data_cfg["input_size"] = (3, cfg.img_size, cfg.img_size)
del _probe
print("data_cfg:", data_cfg)

train_tf = create_transform(**data_cfg, is_training=True,
                            auto_augment="rand-m9-mstd0.5", re_prob=0.25)
val_tf   = create_transform(**data_cfg, is_training=False)

train_ds = ImageFolder(train_dir, transform=train_tf)
val_ds   = ImageFolder(valid_dir, transform=val_tf)
CLASS_TO_IDX = train_ds.class_to_idx
IDX_TO_CLASS = {v:k for k,v in CLASS_TO_IDX.items()}
NUM_CLASSES  = len(CLASS_TO_IDX)
print("classes:", NUM_CLASSES, "| train:", len(train_ds), "| val:", len(val_ds))

## 8. Class imbalance
Compute per-class counts. If the max/min ratio exceeds the trigger, apply **inverse-frequency class weights** in the loss (kept simple and robust; no sampler surprises).

In [ ]:
import numpy as np, torch, collections
counts = collections.Counter([y for _, y in train_ds.samples])
counts = np.array([counts[i] for i in range(NUM_CLASSES)], dtype=np.float64)
ratio = counts.max() / counts.min()
print(f"per-class min={int(counts.min())} max={int(counts.max())} ratio={ratio:.2f}")

if ratio > cfg.imbalance_ratio_trigger:
    weights = counts.sum() / (NUM_CLASSES * counts)          # inverse frequency
    class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
    print("Imbalance detected -> using weighted loss.")
else:
    class_weights = None
    print("Reasonably balanced -> unweighted loss.")

train_loader = DataLoader(train_ds, batch_size=cfg.batch, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=cfg.batch, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True)

## 9. Build model
Pretrained EfficientNet with a fresh classifier head sized to our classes.

In [ ]:
import timm
model = timm.create_model(cfg.m["timm_name"], pretrained=True, num_classes=NUM_CLASSES).to(device)
n_params = sum(p.numel() for p in model.parameters())/1e6
print(f"{cfg.m['timm_name']} | {n_params:.1f}M params | head -> {NUM_CLASSES} classes")

## 10. Training utilities
AMP mixed precision, checkpoint save/load, and **resume** support. Best-by-val-macro-F1 is kept as the master checkpoint.

In [ ]:
import torch, os, time
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import f1_score, accuracy_score

criterion = torch.nn.CrossEntropyLoss(weight=class_weights, label_smoothing=cfg.label_smoothing)
scaler = GradScaler()

def run_epoch(loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    all_y, all_p, tot_loss = [], [], 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.set_grad_enabled(train), autocast():
            out = model(x); loss = criterion(out, y)
        if train:
            optimizer.zero_grad(); scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
        tot_loss += loss.item()*x.size(0)
        all_y += y.tolist(); all_p += out.argmax(1).tolist()
    return (tot_loss/len(loader.dataset),
            accuracy_score(all_y, all_p),
            f1_score(all_y, all_p, average="macro"))

def save_ckpt(path, epoch, optimizer, scheduler, best_f1, phase):
    torch.save({"model": model.state_dict(),
                "optimizer": optimizer.state_dict() if optimizer else None,
                "scheduler": scheduler.state_dict() if scheduler else None,
                "epoch": epoch, "best_f1": best_f1, "phase": phase,
                "class_to_idx": CLASS_TO_IDX, "cfg_model": cfg.m["timm_name"]}, path)

def load_if_exists(path):
    if os.path.exists(path):
        ck = torch.load(path, map_location=device)
        model.load_state_dict(ck["model"]); print(f"resumed <- {path} (epoch {ck['epoch']}, best_f1 {ck.get('best_f1',0):.4f})")
        return ck
    return None

LAST = f"{cfg.ckpt_dir}/last.pt"
BEST = f"{cfg.ckpt_dir}/best.pt"

## 11. Phase A — head warm-up
Freeze the backbone, train only the new classifier head. Fast epochs, stabilises the head before fine-tuning.

In [ ]:
import torch
for p in model.parameters(): p.requires_grad = False
for p in model.get_classifier().parameters(): p.requires_grad = True

optA = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                         lr=cfg.lr_head, weight_decay=cfg.weight_decay)
for ep in range(cfg.epochs_head):
    tl, ta, tf1 = run_epoch(train_loader, optA)
    vl, va, vf1 = run_epoch(val_loader)
    print(f"[HEAD {ep+1}/{cfg.epochs_head}] train f1={tf1:.4f} | val acc={va:.4f} f1={vf1:.4f}")
print("Phase A done.")

## 12. Phase B — full fine-tune
Unfreeze everything, lower LR, **ReduceLROnPlateau** on val macro-F1 (your proven scheduler choice). Checkpoints every epoch to Drive; best-F1 saved separately. Safe to re-run this cell after a disconnect — it resumes from `last.pt`.

In [ ]:
import torch
for p in model.parameters(): p.requires_grad = True

optB = torch.optim.AdamW(model.parameters(), lr=cfg.lr_finetune, weight_decay=cfg.weight_decay)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(optB, mode="max", factor=0.3, patience=2)

start_ep, best_f1 = 0, 0.0
ck = load_if_exists(LAST)
if ck and ck.get("phase") == "B":
    if ck["optimizer"]: optB.load_state_dict(ck["optimizer"])
    if ck["scheduler"]: sched.load_state_dict(ck["scheduler"])
    start_ep, best_f1 = ck["epoch"]+1, ck.get("best_f1", 0.0)

for ep in range(start_ep, cfg.epochs_finetune):
    t0 = time.time()
    tl, ta, tf1 = run_epoch(train_loader, optB)
    vl, va, vf1 = run_epoch(val_loader)
    sched.step(vf1)
    save_ckpt(LAST, ep, optB, sched, best_f1, "B")
    flag = ""
    if vf1 > best_f1:
        best_f1 = vf1; save_ckpt(BEST, ep, optB, sched, best_f1, "B"); flag = "  <-- best"
    print(f"[FT {ep+1}/{cfg.epochs_finetune}] {time.time()-t0:.0f}s | train f1={tf1:.4f} | "
          f"val acc={va:.4f} f1={vf1:.4f} | lr={optB.param_groups[0]['lr']:.1e}{flag}")
print(f"Phase B done. Best val macro-F1 = {best_f1:.4f}")

## 13. Evaluation
Load the best checkpoint, print per-class report + the weakest classes to target next.

In [ ]:
import torch, numpy as np
from sklearn.metrics import classification_report, confusion_matrix
load_if_exists(BEST); model.eval()

ys, ps = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        with autocast(): out = model(x)
        ps += out.argmax(1).cpu().tolist(); ys += y.tolist()

names = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]
rep = classification_report(ys, ps, target_names=names, output_dict=True, zero_division=0)
print(f"Overall accuracy: {rep['accuracy']:.4f} | macro-F1: {rep['macro avg']['f1-score']:.4f}\n")
weak = sorted([(n, rep[n]['f1-score']) for n in names], key=lambda t: t[1])[:8]
print("Weakest classes (target these with more/real-world data):")
for n, f in weak: print(f"  {f:.3f}  {n}")

In [ ]:
import matplotlib.pyplot as plt, numpy as np
cm = confusion_matrix(ys, ps, normalize="true")
plt.figure(figsize=(11,10)); plt.imshow(cm, aspect="auto"); plt.colorbar()
plt.title("Normalized confusion matrix (val)"); plt.xlabel("pred"); plt.ylabel("true")
plt.tight_layout(); plt.savefig(f"{cfg.out_dir}/confusion_matrix.png", dpi=120); plt.show()

## 14. Build the metadata bundle
Writes `class_metadata.json` — the label→(disease, latin_name, mitigations) lookup **plus the exact preprocessing config** the server must replicate. Class order is taken from the trained model's `class_to_idx`, so indices always line up.

In [ ]:
import json

# The curated table (latin names + mitigations) shipped with this notebook.
CURATED = json.loads(r"""{
  "schema_version": 1,
  "model_family": "efficientnet",
  "num_classes": 38,
  "unknown_threshold": 0.4,
  "breakdown_top_n": 3,
  "preprocess": {
    "input_size": 300,
    "mean": [
      0.5,
      0.5,
      0.5
    ],
    "std": [
      0.5,
      0.5,
      0.5
    ],
    "interpolation": "bicubic",
    "resize_mode": "shortest_edge_then_center_crop"
  },
  "classes": [
    {
      "index": 0,
      "folder": "Apple___Apple_scab",
      "crop": "Apple",
      "disease": "Apple Scab",
      "label": "Apple scab",
      "latin_name": "Venturia inaequalis",
      "mitigations": [
        "Remove and destroy infected apple leaves/debris to cut the spore source; avoid overhead irrigation and space plants for airflow.",
        "Apply a labelled protectant fungicide (e.g. captan or a DMI/SDHI) on a preventive schedule during humid weather — confirm product, dosage and pre-harvest interval with a local agronomist.",
        "Rake and destroy fallen leaves in autumn to remove overwintering inoculum; prefer scab-resistant cultivars next planting."
      ]
    },
    {
      "index": 1,
      "folder": "Apple___Black_rot",
      "crop": "Apple",
      "disease": "Black Rot",
      "label": "Black rot",
      "latin_name": "Botryosphaeria obtusa",
      "mitigations": [
        "Prune out cankered wood and mummified fruit well below the infection, and remove them from the orchard.",
        "Apply a labelled fungicide from bloom through the season in wet spells — confirm choice and dosage with an agronomist.",
        "Maintain tree vigour (balanced nutrition, good drainage); avoid bark wounds that let the fungus enter."
      ]
    },
    {
      "index": 2,
      "folder": "Apple___Cedar_apple_rust",
      "crop": "Apple",
      "disease": "Cedar Apple Rust",
      "label": "Cedar apple rust",
      "latin_name": "Gymnosporangium juniperi-virginianae",
      "mitigations": [
        "Where practical, remove nearby juniper/cedar hosts within a few hundred metres to break the two-host cycle.",
        "Apply a preventive fungicide from pink-bud through early fruit set — timing matters more than rate.",
        "Prefer rust-resistant apple cultivars for new plantings."
      ]
    },
    {
      "index": 3,
      "folder": "Apple___healthy",
      "crop": "Apple",
      "disease": "Healthy",
      "label": "Healthy tissue",
      "latin_name": null,
      "mitigations": [
        "No disease detected. Continue routine scouting, balanced fertilisation and proper pruning for airflow.",
        "Keep the orchard floor clear of fallen leaves and mummified fruit as a preventive habit."
      ]
    },
    {
      "index": 4,
      "folder": "Blueberry___healthy",
      "crop": "Blueberry",
      "disease": "Healthy",
      "label": "Healthy tissue",
      "latin_name": null,
      "mitigations": [
        "No disease detected. Maintain acidic soil pH, mulch, and consistent moisture.",
        "Scout weekly during warm humid periods for early leaf spotting."
      ]
    },
    {
      "index": 5,
      "folder": "Cherry_(including_sour)___Powdery_mildew",
      "crop": "Cherry",
      "disease": "Powdery Mildew",
      "label": "Powdery mildew",
      "latin_name": "Podosphaera clandestina",
      "mitigations": [
        "Remove and destroy infected cherry leaves/debris to cut the spore source; avoid overhead irrigation and space plants for airflow.",
        "Apply a labelled sulphur or a labelled DMI fungicide on a preventive schedule during humid weather — confirm product, dosage and pre-harvest interval with a local agronomist.",
        "Prune to open the canopy for airflow and light; avoid excess nitrogen that pushes soft, susceptible growth."
      ]
    },
    {
      "index": 6,
      "folder": "Cherry_(including_sour)___healthy",
      "crop": "Cherry",
      "disease": "Healthy",
      "label": "Healthy tissue",
      "latin_name": null,
      "mitigations": [
        "No disease detected. Keep the canopy open by pruning and avoid overhead wetting.",
        "Scout new shoots for the white powdery growth that signals early mildew."
      ]
    },
    {
      "index": 7,
      "folder": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
      "crop": "Corn",
      "disease": "Gray Leaf Spot",
      "label": "Gray leaf spot",
      "latin_name": "Cercospora zeae-maydis",
      "mitigations": [
        "Rotate away from maize and bury/rotate residue — the fungus overwinters on infected stubble.",
        "Apply a labelled foliar fungicide at early disease onset (around tasseling) if pressure is high; confirm timing with an agronomist.",
        "Prefer gray-leaf-spot-tolerant hybrids and avoid continuous maize on the same field."
      ]
    },
    {
      "index": 8,
      "folder": "Corn_(maize)___Common_rust_",
      "crop": "Corn",
      "disease": "Common Rust",
      "label": "Common rust",
      "latin_name": "Puccinia sorghi",
      "mitigations": [
        "Scout early; most hybrids tolerate light rust without yield loss.",
        "For susceptible seed corn or heavy early infection, apply a labelled fungicide at first pustules.",
        "Prefer rust-resistant hybrids in fields with a history of rust."
      ]
    },
    {
      "index": 9,
      "folder": "Corn_(maize)___Northern_Leaf_Blight",
      "crop": "Corn",
      "disease": "Northern Leaf Blight",
      "label": "Northern leaf blight",
      "latin_name": "Exserohilum turcicum",
      "mitigations": [
        "Rotate crops and incorporate residue to reduce overwintering inoculum.",
        "Apply a labelled fungicide between tasseling and silking if lesions appear on lower leaves early.",
        "Plant NLB-resistant hybrids where the disease recurs."
      ]
    },
    {
      "index": 10,
      "folder": "Corn_(maize)___healthy",
      "crop": "Corn",
      "disease": "Healthy",
      "label": "Healthy tissue",
      "latin_name": null,
      "mitigations": [
        "No disease detected. Maintain balanced fertility and rotate to limit residue-borne diseases.",
        "Scout lower leaves after canopy closure, when foliar diseases usually start."
      ]
    },
    {
      "index": 11,
      "folder": "Grape___Black_rot",
      "crop": "Grape",
      "disease": "Black Rot",
      "label": "Black rot",
      "latin_name": "Guignardia bidwellii",
      "mitigations": [
        "Remove mummified berries and infected canes during dormancy — they carry the fungus to next season.",
        "Begin a preventive fungicide programme from early shoot growth through fruit set in wet weather.",
        "Open the canopy by pruning and leaf-pulling to speed drying after rain."
      ]
    },
    {
      "index": 12,
      "folder": "Grape___Esca_(Black_Measles)",
      "crop": "Grape",
      "disease": "Esca (Black Measles)",
      "label": "Esca (Black Measles)",
      "latin_name": "Phaeomoniella chlamydospora complex",
      "mitigations": [
        "Prune in dry weather and protect large pruning wounds; remove and burn severely affected vines.",
        "There is no reliable spray cure — focus on wound protection, vine vigour and clean planting stock.",
        "Avoid water stress and over-cropping, which worsen symptom expression."
      ]
    },
    {
      "index": 13,
      "folder": "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
      "crop": "Grape",
      "disease": "Isariopsis Leaf Spot",
      "label": "Isariopsis leaf spot",
      "latin_name": "Pseudocercospora vitis",
      "mitigations": [
        "Remove and destroy infected grape leaves/debris to cut the spore source; avoid overhead irrigation and space plants for airflow.",
        "Apply a labelled protectant fungicide (mancozeb or copper-based) on a preventive schedule during humid weather — confirm product, dosage and pre-harvest interval with a local agronomist.",
        "Improve canopy airflow and remove infected leaf litter; the disease rarely needs standalone sprays if black-rot cover is maintained."
      ]
    },
    {
      "index": 14,
      "folder": "Grape___healthy",
      "crop": "Grape",
      "disease": "Healthy",
      "label": "Healthy tissue",
      "latin_name": null,
      "mitigations": [
        "No disease detected. Keep the canopy open and manage humidity around clusters.",
        "Remove mummies and prunings from the vineyard floor as a preventive routine."
      ]
    },
    {
      "index": 15,
      "folder": "Orange___Haunglongbing_(Citrus_greening)",
      "crop": "Orange",
      "disease": "Citrus Greening (HLB)",
      "label": "Citrus greening (HLB)",
      "latin_name": "Candidatus Liberibacter asiaticus",
      "mitigations": [
        "HLB is incurable — remove and destroy confirmed infected trees promptly to protect the rest of the grove.",
        "Control the Asian citrus psyllid vector (Diaphorina citri) with an agronomist-guided programme.",
        "Plant only certified disease-free nursery stock."
      ]
    },
    {
      "index": 16,
      "folder": "Peach___Bacterial_spot",
      "crop": "Peach",
      "disease": "Bacterial Spot",
      "label": "Bacterial spot",
      "latin_name": "Xanthomonas arboricola pv. pruni",
      "mitigations": [
        "Avoid overhead irrigation and working trees when wet; prune for airflow to speed leaf drying.",
        "Apply labelled copper-based sprays on a preventive schedule — bacteria are not curable once established.",
        "Prefer bacterial-spot-tolerant peach cultivars for new plantings."
      ]
    },
    {
      "index": 17,
      "folder": "Peach___healthy",
      "crop": "Peach",
      "disease": "Healthy",
      "label": "Healthy tissue",
      "latin_name": null,
      "mitigations": [
        "No disease detected. Maintain tree vigour and avoid overhead wetting of foliage.",
        "Scout leaves and fruit for small dark angular spots during warm wet spells."
      ]
    },
    {
      "index": 18,
      "folder": "Pepper,_bell___Bacterial_spot",
      "crop": "Bell Pepper",
      "disease": "Bacterial Spot",
      "label": "Bacterial spot",
      "latin_name": "Xanthomonas euvesicatoria",
      "mitigations": [
        "Use certified disease-free seed/transplants; rogue out early infected plants.",
        "Avoid overhead irrigation and handling plants when wet; apply labelled copper (often with mancozeb) preventively.",
        "Rotate away from peppers/tomatoes for 2-3 years and remove crop debris after harvest."
      ]
    },
    {
      "index": 19,
      "folder": "Pepper,_bell___healthy",
      "crop": "Bell Pepper",
      "disease": "Healthy",
      "label": "Healthy tissue",
      "latin_name": null,
      "mitigations": [
        "No disease detected. Keep foliage dry and rotate away from solanaceous crops.",
        "Scout for small water-soaked leaf spots after warm, rainy periods."
      ]
    },
    {
      "index": 20,
      "folder": "Potato___Early_blight",
      "crop": "Potato",
      "disease": "Early Blight",
      "label": "Early blight",
      "latin_name": "Alternaria solani",
      "mitigations": [
        "Remove volunteer plants and debris; keep plants vigorous — early blight hits stressed, older foliage first.",
        "Apply a labelled protectant fungicide (e.g. chlorothalonil or mancozeb) on a preventive schedule in warm humid weather.",
        "Rotate out of potatoes/tomatoes for 2 years and avoid overhead irrigation late in the day."
      ]
    },
    {
      "index": 21,
      "folder": "Potato___Late_blight",
      "crop": "Potato",
      "disease": "Late Blight",
      "label": "Late blight",
      "latin_name": "Phytophthora infestans",
      "mitigations": [
        "Act fast — late blight spreads explosively in cool wet weather. Destroy infected plants and cull infected tubers.",
        "Apply a labelled late-blight fungicide preventively before/at first sign; switch to systemic actives under high pressure (agronomist-guided).",
        "Plant certified seed, hill soil over tubers, and avoid overhead irrigation."
      ]
    },
    {
      "index": 22,
      "folder": "Potato___healthy",
      "crop": "Potato",
      "disease": "Healthy",
      "label": "Healthy tissue",
      "latin_name": null,
      "mitigations": [
        "No disease detected. Use certified seed and scout closely during cool, wet spells for late blight.",
        "Hill soil over developing tubers to protect them from spores."
      ]
    },
    {
      "index": 23,
      "folder": "Raspberry___healthy",
      "crop": "Raspberry",
      "disease": "Healthy",
      "label": "Healthy tissue",
      "latin_name": null,
      "mitigations": [
        "No disease detected. Thin canes for airflow and keep the row base clear of debris.",
        "Scout new canes for leaf spotting and rust during humid weather."
      ]
    },
    {
      "index": 24,
      "folder": "Soybean___healthy",
      "crop": "Soybean",
      "disease": "Healthy",
      "label": "Healthy tissue",
      "latin_name": null,
      "mitigations": [
        "No disease detected. Maintain rotation and scout for foliar spotting through the canopy.",
        "Manage residue and volunteers to limit carry-over of foliar diseases."
      ]
    },
    {
      "index": 25,
      "folder": "Squash___Powdery_mildew",
      "crop": "Squash",
      "disease": "Powdery Mildew",
      "label": "Powdery mildew",
      "latin_name": "Podosphaera xanthii",
      "mitigations": [
        "Scout undersides of older leaves; remove the most heavily coated leaves to slow spread.",
        "Apply sulphur, potassium bicarbonate, or a labelled fungicide at first white colonies — rotate actives to avoid resistance.",
        "Space plants and prefer mildew-tolerant varieties for airflow and durability."
      ]
    },
    {
      "index": 26,
      "folder": "Strawberry___Leaf_scorch",
      "crop": "Strawberry",
      "disease": "Leaf Scorch",
      "label": "Leaf scorch",
      "latin_name": "Diplocarpon earlianum",
      "mitigations": [
        "Remove and destroy infected strawberry leaves/debris to cut the spore source; avoid overhead irrigation and space plants for airflow.",
        "Apply a labelled protectant fungicide (captan or a labelled option) on a preventive schedule during humid weather — confirm product, dosage and pre-harvest interval with a local agronomist.",
        "Renovate beds after harvest, remove old infected leaves, and avoid overhead irrigation."
      ]
    },
    {
      "index": 27,
      "folder": "Strawberry___healthy",
      "crop": "Strawberry",
      "disease": "Healthy",
      "label": "Healthy tissue",
      "latin_name": null,
      "mitigations": [
        "No disease detected. Keep foliage dry with drip irrigation and remove old leaves at renovation.",
        "Scout for purple-bordered leaf spots after warm, wet weather."
      ]
    },
    {
      "index": 28,
      "folder": "Tomato___Bacterial_spot",
      "crop": "Tomato",
      "disease": "Bacterial Spot",
      "label": "Bacterial spot",
      "latin_name": "Xanthomonas spp.",
      "mitigations": [
        "Use certified clean seed/transplants; avoid overhead irrigation and handling wet plants.",
        "Apply labelled copper (often tank-mixed with mancozeb) preventively — copper alone weakens under resistance.",
        "Rotate 2-3 years away from tomato/pepper and remove debris after harvest."
      ]
    },
    {
      "index": 29,
      "folder": "Tomato___Early_blight",
      "crop": "Tomato",
      "disease": "Early Blight",
      "label": "Early blight",
      "latin_name": "Alternaria solani",
      "mitigations": [
        "Remove lower infected leaves, mulch to block soil splash, and stake plants for airflow.",
        "Apply a labelled protectant fungicide (chlorothalonil/mancozeb) preventively in warm humid weather.",
        "Rotate out of solanaceous crops for 2 years and keep plants well nourished."
      ]
    },
    {
      "index": 30,
      "folder": "Tomato___Late_blight",
      "crop": "Tomato",
      "disease": "Late Blight",
      "label": "Late blight",
      "latin_name": "Phytophthora infestans",
      "mitigations": [
        "Move fast — destroy infected plants immediately; the disease can wipe out a field in days in cool wet weather.",
        "Apply a labelled late-blight fungicide preventively; use systemic actives under high pressure (agronomist-guided).",
        "Avoid overhead irrigation, space plants, and never compost infected material."
      ]
    },
    {
      "index": 31,
      "folder": "Tomato___Leaf_Mold",
      "crop": "Tomato",
      "disease": "Leaf Mold",
      "label": "Leaf mold",
      "latin_name": "Passalora fulva",
      "mitigations": [
        "Primarily a greenhouse/high-humidity disease — improve ventilation and reduce humidity below 85%.",
        "Remove affected leaves and apply a labelled fungicide if it persists; avoid overhead wetting.",
        "Prefer leaf-mold-resistant cultivars in protected culture."
      ]
    },
    {
      "index": 32,
      "folder": "Tomato___Septoria_leaf_spot",
      "crop": "Tomato",
      "disease": "Septoria Leaf Spot",
      "label": "Septoria leaf spot",
      "latin_name": "Septoria lycopersici",
      "mitigations": [
        "Remove and destroy infected tomato leaves/debris to cut the spore source; avoid overhead irrigation and space plants for airflow.",
        "Apply a labelled protectant fungicide (chlorothalonil or mancozeb) on a preventive schedule during humid weather — confirm product, dosage and pre-harvest interval with a local agronomist.",
        "Mulch to stop soil splash, remove infected lower leaves early, and rotate away from tomato for 2 years."
      ]
    },
    {
      "index": 33,
      "folder": "Tomato___Spider_mites Two-spotted_spider_mite",
      "crop": "Tomato",
      "disease": "Two-Spotted Spider Mite",
      "label": "Spider mites",
      "latin_name": "Tetranychus urticae",
      "mitigations": [
        "This is a pest, not a fungus — check leaf undersides for fine webbing and stippling; hose off light infestations.",
        "Conserve/introduce predatory mites; use a labelled miticide only when needed and rotate actives to avoid resistance.",
        "Avoid dusty, water-stressed conditions, which favour mite outbreaks."
      ]
    },
    {
      "index": 34,
      "folder": "Tomato___Target_Spot",
      "crop": "Tomato",
      "disease": "Target Spot",
      "label": "Target spot",
      "latin_name": "Corynespora cassiicola",
      "mitigations": [
        "Remove and destroy infected tomato leaves/debris to cut the spore source; avoid overhead irrigation and space plants for airflow.",
        "Apply a labelled protectant fungicide (chlorothalonil or a labelled SDHI) on a preventive schedule during humid weather — confirm product, dosage and pre-harvest interval with a local agronomist.",
        "Improve airflow with staking/pruning, remove infected debris, and avoid prolonged leaf wetness."
      ]
    },
    {
      "index": 35,
      "folder": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
      "crop": "Tomato",
      "disease": "Yellow Leaf Curl Virus",
      "label": "Yellow leaf curl virus",
      "latin_name": "Tomato yellow leaf curl virus (Begomovirus)",
      "mitigations": [
        "The virus is incurable — rogue out infected plants and control the whitefly vector (Bemisia tabaci).",
        "Use resistant/tolerant varieties and protect young transplants with nets or reflective mulch.",
        "Manage weeds that host whiteflies and avoid staggered plantings that bridge the vector."
      ]
    },
    {
      "index": 36,
      "folder": "Tomato___Tomato_mosaic_virus",
      "crop": "Tomato",
      "disease": "Tomato Mosaic Virus",
      "label": "Tomato mosaic virus",
      "latin_name": "Tomato mosaic virus (ToMV)",
      "mitigations": [
        "Remove and destroy infected plants; the virus spreads by handling, tools and debris (no vector needed).",
        "Wash hands and disinfect tools between plants; avoid tobacco use around plants.",
        "Use certified seed and resistant cultivars; the virus survives long-term in debris."
      ]
    },
    {
      "index": 37,
      "folder": "Tomato___healthy",
      "crop": "Tomato",
      "disease": "Healthy",
      "label": "Healthy tissue",
      "latin_name": null,
      "mitigations": [
        "No disease detected. Stake and mulch plants, water at the base, and scout lower leaves weekly.",
        "Rotate out of solanaceous crops each season to limit soil-borne carry-over."
      ]
    }
  ]
}""")
by_folder = {c["folder"]: c for c in CURATED["classes"]}

missing = [f for f in CLASS_TO_IDX if f not in by_folder]
assert not missing, f"metadata missing for: {missing}"

classes_out = []
for folder, idx in sorted(CLASS_TO_IDX.items(), key=lambda kv: kv[1]):
    c = by_folder[folder]
    classes_out.append({
        "index": idx, "folder": folder, "crop": c["crop"], "disease": c["disease"],
        "label": c["label"], "latin_name": c["latin_name"], "mitigations": c["mitigations"],
    })

mean = data_cfg["mean"]; std = data_cfg["std"]
bundle = {
    "schema_version": 1,
    "model_family": cfg.MODEL_KEY,
    "timm_name": cfg.m["timm_name"],
    "num_classes": NUM_CLASSES,
    "unknown_threshold": CURATED["unknown_threshold"],
    "breakdown_top_n": CURATED["breakdown_top_n"],
    "preprocess": {
        "input_size": cfg.img_size,
        "mean": [float(x) for x in mean],
        "std":  [float(x) for x in std],
        "interpolation": data_cfg.get("interpolation", "bicubic"),
        "crop_pct": float(data_cfg.get("crop_pct", 1.0) or 1.0),
    },
    "classes": classes_out,
}
with open(f"{cfg.out_dir}/class_metadata.json","w") as f:
    json.dump(bundle, f, indent=2, ensure_ascii=False)
print("wrote", f"{cfg.out_dir}/class_metadata.json | preprocess:", bundle["preprocess"])

## 15. Export to ONNX
Opset 17 (compatible with `onnxruntime==1.19.2`), dynamic batch axis, simplified with `onnxsim`. We then **verify parity**: onnxruntime output must match torch within tolerance, or the export is rejected.

In [ ]:
import torch, numpy as np, onnx, onnxruntime as ort
from onnxsim import simplify

model.eval()
onnx_path = f"{cfg.out_dir}/disease_model.onnx"
dummy = torch.randn(1, 3, cfg.img_size, cfg.img_size, device=device)

torch.onnx.export(
    model, dummy, onnx_path,
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17, do_constant_folding=True,
)
m = onnx.load(onnx_path); ms, ok = simplify(m)
assert ok, "onnxsim failed"; onnx.save(ms, onnx_path)

# parity check
with torch.no_grad():
    t_out = model(dummy).cpu().numpy()
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
o_out = sess.run(None, {"input": dummy.cpu().numpy()})[0]
max_diff = float(np.abs(t_out - o_out).max())
print(f"ONNX parity max |diff| = {max_diff:.2e}")
assert max_diff < 1e-3, "ONNX output diverges from torch — do not ship"
print("ONNX export verified ->", onnx_path)

## 16. Package the `model/` bundle
Zips `disease_model.onnx` + `class_metadata.json` for download and leaves a copy on Drive.

In [ ]:
import shutil, os
bundle_dir = f"{cfg.out_dir}/bundle"; os.makedirs(bundle_dir, exist_ok=True)
shutil.copy(f"{cfg.out_dir}/disease_model.onnx", bundle_dir)
shutil.copy(f"{cfg.out_dir}/class_metadata.json", bundle_dir)
zip_base = f"{cfg.drive_root}/disease_model_{cfg.MODEL_KEY}_bundle"
shutil.make_archive(zip_base, "zip", bundle_dir)
print("bundle zip:", zip_base + ".zip")
try:
    from google.colab import files; files.download(zip_base + ".zip")
except Exception as e:
    print("download from the Drive path above:", e)

## 17. Sanity test — prove it matches the backend contract
Loads only the ONNX + metadata (no torch), runs a real val image, and prints an `InferenceResult`-shaped dict: `disease`, `latin_name`, `confidence_pct`, `breakdown` (top-N + "Other / unknown"), `mitigations`. This is exactly what your `OnnxInferenceProvider` will return.

In [ ]:
import onnxruntime as ort, numpy as np, json, random
from PIL import Image

meta = json.load(open(f"{cfg.out_dir}/class_metadata.json"))
pp = meta["preprocess"]; classes = meta["classes"]
sess = ort.InferenceSession(f"{cfg.out_dir}/disease_model.onnx", providers=["CPUExecutionProvider"])

def preprocess(img: Image.Image):
    s = pp["input_size"]
    img = img.convert("RGB").resize((s, s), Image.BICUBIC)   # provider mirrors this
    a = np.asarray(img, np.float32)/255.0
    a = (a - np.array(pp["mean"], np.float32)) / np.array(pp["std"], np.float32)
    return a.transpose(2,0,1)[None].astype(np.float32)

def softmax(z): e = np.exp(z - z.max()); return e/e.sum()

def classify(img):
    logits = sess.run(None, {"input": preprocess(img)})[0][0]
    probs = softmax(logits); top = int(probs.argmax()); conf = float(probs[top])
    order = probs.argsort()[::-1][:meta["breakdown_top_n"]]
    breakdown = [{"label": classes[i]["label"], "pct": round(float(probs[i])*100,1)} for i in order]
    other = round(max(0.0, 1 - sum(p["pct"] for p in breakdown)/100)*100, 1)
    breakdown.append({"label": "Other / unknown", "pct": other})
    if conf < meta["unknown_threshold"]:
        return {"disease": "Uncertain — retake photo / consult expert", "latin_name": None,
                "confidence_pct": round(conf*100,1), "breakdown": breakdown,
                "mitigations": ["Low confidence. Retake a sharp, well-lit close-up of a single affected leaf.",
                                "If it recurs, consult your local agronomist with the photo."],
                "demo_mode": False}
    c = classes[top]
    return {"disease": c["disease"], "latin_name": c["latin_name"],
            "confidence_pct": round(conf*100,1), "breakdown": breakdown,
            "mitigations": c["mitigations"], "demo_mode": False}

path, y = random.choice(val_ds.samples)
print("true:", IDX_TO_CLASS[y])
print(json.dumps(classify(Image.open(path)), indent=2, ensure_ascii=False))

## 18. Next steps
1. **Drop into the backend** — copy `disease_model.onnx` + `class_metadata.json` to `app/services/scanner/model/`, add `OnnxInferenceProvider` (provided separately), set `INFERENCE_PROVIDER=onnx`. See `BACKEND_INTEGRATION.md`.
2. **Real-world robustness** — the weak classes from §13 are where lab-clean training hurts. Collect a few hundred real field photos (or fold in **PlantDoc**) and fine-tune further; that's what actually moves edge-case accuracy.
3. **Escalate to B4** only if the accuracy gap justifies the extra hours — flip `MODEL_KEY="b4"` and re-run.
4. **TFLite later** — ONNX keeps that path open (`onnx2tf`), so no retrain needed if you go on-device.
